# ROBERTA SENTIMENT

In [1]:
!pip install -q transformers torch tqdm

In [2]:
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from transformers import pipeline

In [3]:
df = pd.read_csv("Model Rule Based.csv")
df

,Channel_Sumber,Author,Comment_Final,Likes,Time,Tipe_Komentar,Sentiment_Rule
0,Jagat Review,@CheverrlynAdyba,link beli chamon pro 5g,0,2026-03-21 13:04:52+00:00,Utama,Netral
1,Jagat Review,@caesarrizkie9742,ngomongin rog kah kamera turun lihat review ca...,0,2026-03-17 20:41:02+00:00,Utama,Positif
2,Jagat Review,@adealfian664,user samsung one ui ui enak ui ui if you can t...,0,2026-03-13 23:27:54+00:00,Utama,Netral
3,Jagat Review,@RsyatLyioz,tecno pova ultra 5g,0,2026-03-12 20:11:30+00:00,Utama,Netral
4,Jagat Review,@CitraUtut,tolong buatin rekomendasi lebaran,2,2026-03-02 23:12:23+00:00,Utama,Positif
...,...,...,...,...,...,...,...
7194,Gadgetin,@imnaps,akhirnyaa rekomendasi,1,2025-12-07 09:27:42+00:00,Utama,Positif
7195,Gadgetin,@MOLEND95,first wkwkkw,0,2025-12-07 09:27:40+00:00,Utama,Netral
7196,Gadgetin,@m.abidalbaqie9527,party popper party popper,0,2025-12-07 09:27:40+00:00,Utama,Netral
7197,Gadgetin,@tayaaasy8007,sampe skrng kebel loudly crying face loudly cr...,0,2025-12-07 09:27:39+00:00,Utama,Netral


In [4]:
df.columns.tolist()

['Channel_Sumber',
 'Author',
 'Comment_Final',
 'Likes',
 'Time',
 'Tipe_Komentar',
 'Sentiment_Rule']

In [5]:
device = 0 if torch.cuda.is_available() else -1
print("Device:", device)

Device: -1


## Load Model RoBerta

In [6]:
model = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

print("Model berhasil di-load")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model berhasil di-load


In [7]:
model("kamera bagus banget")

[{'label': 'positive', 'score': 0.9956856966018677}]

## Prediksi Batch

In [8]:
from tqdm import tqdm
import torch

device = 0 if torch.cuda.is_available() else -1

komentar = (
    df["Comment_Final"]
    .astype(str)
    .fillna("")
    .tolist()
)

In [9]:
hasil = []

batch_size = 32

for i in tqdm(range(0, len(komentar), batch_size)):

    batch = komentar[i:i+batch_size]

    pred = model(
        batch,
        truncation=True,
        max_length=512
    )

    hasil.extend(
        [x["label"] for x in pred]
    )

df["Sentiment_Roberta"] = hasil

print("✅ Prediksi selesai")

100%|██████████| 225/225 [14:35<00:00,  3.89s/it]

✅ Prediksi selesai


In [10]:
df[
    ["Comment_Final","Sentiment_Roberta"]
].sample(10)

,Comment_Final,Sentiment_Roberta
2470,iya pas ngomong kamera utama tele 4k60 edit te...,negative
3323,yahh hot pro masuk,negative
3737,review handphone advan,neutral
5207,alhamdulillah tonton pakai handphone juaraa ka...,positive
371,moto edge fusion,neutral
594,a56 5jt,negative
2116,moga abang sehat butuh banget konten kreator e...,positive
2623,kakak rekomendasi lebaran,neutral
4293,poco f7 pro beli wkwkwk,neutral
5026,tonton pakai handphone x7 pro ya lumayan ketim...,positive


In [11]:
df["Sentiment_Roberta"].value_counts()

,count
Sentiment_Roberta,
neutral,4214
negative,1663
positive,1322


## Ubah label Inggris → Indonesia

In [12]:
map_label = {
    "positive": "Positif",
    "negative": "Negatif",
    "neutral": "Netral"
}

df["Sentiment_Roberta"] = (
    df["Sentiment_Roberta"]
    .replace(map_label)
)

In [13]:
df[["Comment_Final","Sentiment_Rule","Sentiment_Roberta"]].sample(10)

,Comment_Final,Sentiment_Rule,Sentiment_Roberta
2683,berkat channel gadgetin review david milih bel...,Netral,Netral
3388,serius beli handphone susah harga ram btw beli...,Netral,Negatif
3746,rekomendasi laptop nominasi 10jt,Positif,Netral
601,rekomendasi kamera juta grinning face with smi...,Positif,Netral
5060,sesuai duga handphone pilih juara tahan kelas ...,Positif,Netral
5517,bikin kategori tablet folded hands,Netral,Netral
2712,david poco x7 kek gitu gampang panas trus mato...,Negatif,Netral
1761,thanks a lot review bantu milih handphone sesu...,Positif,Positif
1865,hasil kamera apparture nilai ukur lensa pixel ...,Netral,Netral
2245,bagus abangkuhh bagus bangett party popper,Positif,Positif


## Crosstab Rule vs RoBERTa

In [14]:
ct = pd.crosstab(
    df["Sentiment_Rule"],
    df["Sentiment_Roberta"]
)

display(ct)

Sentiment_Roberta,Negatif,Netral,Positif
Sentiment_Rule,,,
Negatif,137,53,24
Netral,1329,3589,561
Positif,197,572,737


In [15]:
ct_percent = pd.crosstab(
    df["Sentiment_Rule"],
    df["Sentiment_Roberta"],
    normalize="index"
) * 100

ct_final = (
    ct.astype(str)
    + " ("
    + ct_percent.round(1).astype(str)
    + "%)"
)

display(ct_final)

Sentiment_Roberta,Negatif,Netral,Positif
Sentiment_Rule,,,
Negatif,137 (64.0%),53 (24.8%),24 (11.2%)
Netral,1329 (24.3%),3589 (65.5%),561 (10.2%)
Positif,197 (13.1%),572 (38.0%),737 (48.9%)


## Distribusi Sentimen

In [16]:
print("=== RULE BASED ===")
display(df["Sentiment_Rule"].value_counts())

=== RULE BASED ===


,count
Sentiment_Rule,
Netral,5479
Positif,1506
Negatif,214


In [17]:
print("=== ROBERTA ===")
display(df["Sentiment_Roberta"].value_counts())

=== ROBERTA ===


,count
Sentiment_Roberta,
Netral,4214
Negatif,1663
Positif,1322


In [18]:
df.to_csv(
    "Youtube Sentiment_Roberta.csv",
    index=False,
    encoding="utf-8-sig")

print("Youtube Sentiment_Roberta.csv tersimpan")

Youtube Sentiment_Roberta.csv tersimpan
